In [ ]:
import gymnasium as gym
import tensorflow as tf
import numpy as np
from keras.models import load_model  , Model
from keras.layers import Dense, Input
from keras.losses import MeanSquaredError
from keras.models import clone_model


env = gym.make("CartPole-v1")

net_input = Input(shape=(4,))
x = Dense(64 , activation='relu')(net_input)
x = Dense(32 , activation='relu')(x)
net_output = Dense(2 , activation='linear')(x)
q_net = Model(inputs=net_input , outputs=net_output)
q_net.compile(optimizer='adam')
loss_fn = MeanSquaredError()

target_net = clone_model(q_net)

#PARAMETS
EPSILON = 0.1
EPSILON_DECAY = 1.005
GAMMA = 0.99
NUM_EPISODES = 300


# ==========================================
# Policy Function
# ==========================================
def policy(state, explore=0.0):
    # Epsilon-Greedy policy
    if tf.random.uniform(shape=(), maxval=1.0) <= explore:
        return np.random.randint(0, 2)
    
    # State already (1, 4) form mein aayega
    q_values = q_net(state)
    return int(tf.argmax(q_values[0]).numpy())



for episode in range(NUM_EPISODES):
    state, _ = env.reset()
    state = tf.convert_to_tensor(state.reshape(1, 4), dtype=tf.float32)
    done = False
    total_reward = 0

    while not done:
        action = policy(state, EPSILON)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state_tensor = tf.convert_to_tensor(next_state.reshape(1, 4), dtype=tf.float32)

        # Q-learning update
        target_q_values = target_net(next_state_tensor)
        max_target_q_value = tf.reduce_max(target_q_values[0])
        target_value = reward + (GAMMA * max_target_q_value * (1 - int(done)))

        with tf.GradientTape() as tape:
            q_values = q_net(state)
            q_value = q_values[0][action]
            loss = loss_fn(tf.expand_dims(target_value, 0), tf.expand_dims(q_value, 0))

        grads = tape.gradient(loss, q_net.trainable_variables)
        q_net.optimizer.apply_gradients(zip(grads, q_net.trainable_variables))

        state = next_state_tensor
        total_reward += reward

    # Update target network every episode
    target_net.set_weights(q_net.get_weights())

    # Decay epsilon
    EPSILON /= EPSILON_DECAY

    print(f"Episode {episode + 1}: Total Reward: {total_reward}, Epsilon: {EPSILON:.4f}")